# Modelling

The objective of this stage is to establish forecasting baselines and compare regression models for predicting energy consumption 15 minutes ahead. Models are evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE), with lower values indicating better predictive performance.

In [53]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

## Load Processed Data

In [ ]:
train = pd.read_csv('../data/train.csv', parse_dates=['date'])
val = pd.read_csv('../data/validation.csv', parse_dates=['date'])
test = pd.read_csv('../data/test.csv', parse_dates=['date'])

## Naive Baseline

A naive lag-1 forecast uses the previous 15-minute energy consumption as the prediction for the current interval. This provides a simple benchmark against which more complex models can be evaluated.

In [10]:
# assumption baseline
baseline_mae = mean_absolute_error(val['Usage_kWh'], val['lag_1'])
baseline_rmse = np.sqrt(mean_squared_error(val['Usage_kWh'], val['lag_1']))
print(f'Baseline MAE: {baseline_mae}')
print(f'Baseline RMSE: {baseline_rmse}')                      

Baseline MAE: 5.7516411251212425
Baseline RMSE: 12.78130286585032


## Feature Selection and Preprocessing

The forecasting features consist of time-based variables, categorical operating indicators, and historical energy consumption through lag features. The date column is retained for chronological reference and plotting but is not used as a model input.

In [51]:
# separating predictors from target
x_train = train[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_train = train['Usage_kWh']

x_val = val[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_val = val['Usage_kWh']

x_test = test[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_test = test['Usage_kWh']


## Linear Regression

Linear Regression was used as a simple machine-learning benchmark to assess whether the selected features can explain future energy consumption through a linear relationship.

In [36]:
# one-hot encoding categorical features
categorical_features = ['Day_of_week','WeekStatus']
numerical_features = ['hour','minute','month','lag_1','lag_2','lag_4','lag_96','lag_672']

# creating a preprocessor
preprocessor = ColumnTransformer(transformers = [('categorical', OneHotEncoder(handle_unknown='ignore'),
                                                  categorical_features),
                                                 ('numerical', 'passthrough', numerical_features)])

# model pipeline with linear regression
linear_pipeline = Pipeline(steps = [('preprocessor', preprocessor),
                                 ('linearmodel', LinearRegression())])

# training model
linear_pipeline.fit(x_train, y_train)
# predict on validation set
y_pred = linear_pipeline.predict(x_val)

# evaluate
linear_mae = mean_absolute_error(y_val, y_pred)
linear_rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'Linear Regression MAE: {linear_mae}')
print(f'Linear Regression RMSE: {linear_rmse}')

Linear Regression MAE: 6.584016728578933
Linear Regression RMSE: 11.931708119177065


## Random Forest

Random Forest was selected to capture nonlinear relationships and interactions between the forecasting features that may not be represented by the linear model.

In [ ]:
rf_pipeline = Pipeline(steps = [('preprocessor', preprocessor),
                                 ('rfmodel', RandomForestRegressor(random_state=42))])
rf_pipeline.fit(x_train, y_train)
y_pred = rf_pipeline.predict(x_val)

# evaluate
rf_mae = mean_absolute_error(y_val, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'Random Forest MAE: {rf_mae}')
print(f'Random Forest RMSE: {rf_rmse}')

Random Forest MAE: 4.806590184287099
Random Forest RMSE: 10.013699178062296


## Model Comparison

The initial models are compared using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). Lower values indicate better predictive performance.

## Random Forest Hyperparameter Tuning

TimeSeriesSplit was used for cross-validation to preserve the temporal ordering of the observations. The hyperparameter search was optimized using negative Mean Absolute Error, meaning that the configuration with the lowest validation MAE was selected.

In [45]:
tscv = TimeSeriesSplit(n_splits = 5)

param_grid = {'rfmodel__n_estimators': [100, 200],
              'rfmodel__max_depth': [10,20,30],
              'rfmodel__min_samples_leaf': [1,2,5],
              'rfmodel__min_samples_split': [2,5]}

grid_search = GridSearchCV(estimator = rf_pipeline,
                           param_grid = param_grid,
                           cv = tscv,
                           scoring = 'neg_mean_absolute_error',
                           n_jobs = -1,
                           verbose = 1)

grid_search.fit(x_train, y_train)

print('Best Parameters:')
print(grid_search.best_params_)

print('\nBest CV MAE:')
print(-grid_search.best_score_)
              

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters:
{'rfmodel__max_depth': 30, 'rfmodel__min_samples_leaf': 5, 'rfmodel__min_samples_split': 5, 'rfmodel__n_estimators': 200}

Best CV MAE:
5.408297880128396


## Tuned Model Evaluation

In [ ]:
best_rf = grid_search.best_estimator_

y_val_pred = best_rf.predict(x_val)

val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Tuned RF Validation MAE:", val_mae)
print("Tuned RF Validation RMSE:", val_rmse)

Tuned Random Forest MAE: 4.547796103539404
Tuned Random Forest RMSE: 9.796360303131697


## Overfitting Check

Training performance is compared with validation performance to assess whether the tuned model shows evidence of overfitting.

In [ ]:
# checking model overfitting
y_train_pred = best_rf.predict(x_train)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

print(f'Tuned Random Forest MAE: {train_mae}')
print(f'Tuned Random Forest RMSE: {train_rmse}')

Tuned Random Forest MAE: 3.3821681720879417
Tuned Random Forest RMSE: 7.199682961017556


## Test Set Evaluation

The test set provides the final evaluation of the selected model on observations that were not used during model training or hyperparameter selection.

In [ ]:
y_test_pred = best_rf.predict(x_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f'Tuned Random Forest MAE: {test_mae}')
print(f'Tuned Random Forest RMSE: {test_rmse}')

Tuned Random Forest MAE: 4.40803393219915
Tuned Random Forest RMSE: 9.393289612608767


## Save Final Model

The selected tuned Random Forest pipeline is saved for later use in model interpretation and application development.

In [ ]:
joblib.dump(best_rf, '../models/final_random_forest.pkl')

['final_random_forest.pkl']

Summary: Several forecasting models were developed to predict 15-minute-ahead energy usage using historical usage and time-based features. A naïve persistence baseline was first established, achieving an MAE of 5.75 kWh and RMSE of 12.78 kWh. Linear Regression was then tested but performed worse in MAE, while a Random Forest Regressor performed better than the baseline. Time-series cross-validation was used to tune the Random Forest while preserving the chronological structure of the data and avoiding future-data leakage. The final tuned model used 200 trees, a maximum depth of 30, a minimum of 5 samples per leaf, and a minimum of 5 samples for splitting. An overfitting check showed higher training performance than validation performance but no indication of severe overfitting. Finally, the model was evaluated on the previously untouched test set, achieving a MAE of 4.41 kWh and RMSE of 9.39 kWh, representing approximately 23% and 26% improvements over the naïve baseline respectively.
